# 🎮 Clustering Perilaku Pemain Game Online
## Menggunakan K-Means Clustering dengan Framework CRISP-DM

---

| Info | Detail |
|------|--------|
| **Nama** | louishasashi-dev |
| **Email** | louishasashi@gmail.com |
| **Program Studi** | Sistem Informasi |
| **Institusi** | Institut Teknologi & Bisnis Bina Sarana Global |
| **Mata Kuliah** | Konsep Data Warehouse & Mining |
| **Metode** | Clustering (K-Means) |
| **Dataset** | Online Gaming Behavior Dataset (Kaggle) |
| **Link Dataset** | https://www.kaggle.com/datasets/rabieelkharoua/predict-online-gaming-behavior-dataset |


---
# 1. Business Understanding


## 1.1 Latar Belakang Masalah

Industri game online terus berkembang pesat dengan jutaan pemain aktif di seluruh dunia.
Platform game membutuhkan pemahaman mendalam tentang perilaku pemain untuk meningkatkan
retensi, personalisasi pengalaman bermain, dan strategi monetisasi yang tepat.

Permasalahan utama yang dihadapi:
- Developer game kesulitan memahami pola perilaku pemain yang beragam
- Tidak ada segmentasi yang jelas antara tipe-tipe pemain
- Strategi engagement yang bersifat *one-size-fits-all* kurang efektif


## 1.2 Tujuan Project

1. Mengelompokkan pemain game online ke dalam segmen berdasarkan perilaku bermain mereka
2. Mengidentifikasi karakteristik unik setiap kelompok pemain
3. Memberikan rekomendasi strategi engagement untuk masing-masing segmen


## 1.3 Manfaat Analisis

- **Bagi Developer Game:** Memahami segmen pemain untuk merancang fitur yang lebih relevan
- **Bagi Tim Marketing:** Menyusun strategi promosi yang tertarget per segmen
- **Bagi Desainer UX:** Menyesuaikan pengalaman bermain sesuai profil pengguna
- **Bagi Bisnis:** Meningkatkan retensi pemain dan pendapatan in-game


---
# 2. Data Understanding


## 2.1 Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
print('Libraries berhasil diimport ✅')

## 2.2 Load Dataset

In [ ]:
df = pd.read_csv('online_gaming_behavior_insights.csv')

print(f'Jumlah baris  : {df.shape[0]:,}')
print(f'Jumlah kolom  : {df.shape[1]}')
print()
df.head(10)

## 2.3 Deskripsi Dataset

In [ ]:
# Deskripsi tiap kolom
desc = {
    'PlayerID': 'ID unik pemain',
    'Age': 'Usia pemain (tahun)',
    'Gender': 'Jenis kelamin pemain',
    'Location': 'Lokasi/negara pemain',
    'GameGenre': 'Genre game yang dimainkan',
    'PlayTimeHours': 'Total jam bermain per sesi rata-rata',
    'InGamePurchases': 'Apakah melakukan pembelian in-game (0/1)',
    'GameDifficulty': 'Tingkat kesulitan game yang dipilih',
    'SessionsPerWeek': 'Jumlah sesi bermain per minggu',
    'AvgSessionDurationMinutes': 'Durasi rata-rata per sesi (menit)',
    'PlayerLevel': 'Level pemain saat ini',
    'AchievementsUnlocked': 'Jumlah pencapaian yang dibuka',
    'EngagementLevel': 'Tingkat keterlibatan pemain (Low/Medium/High)'
}
print('Deskripsi Kolom Dataset:')
print('-' * 55)
for col, d in desc.items():
    print(f'  {col:<30} : {d}')

## 2.4 Eksplorasi Data Awal

In [ ]:
print('=== Info Dataset ===')
df.info()
print()
print('=== Statistik Deskriptif ===')
df.describe().round(2)

In [ ]:
print('=== Distribusi Variabel Kategorikal ===')
for col in ['Gender', 'Location', 'GameGenre', 'GameDifficulty', 'EngagementLevel']:
    print(f'\n{col}:')
    print(df[col].value_counts().to_string())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Distribusi Fitur Perilaku Bermain', fontsize=16, fontweight='bold', y=1.01)

features_plot = ['PlayTimeHours', 'SessionsPerWeek', 'AvgSessionDurationMinutes',
                  'PlayerLevel', 'AchievementsUnlocked', 'Age']
colors = ['#4A90E2','#50C878','#FF6B6B','#FFD700','#9B59B6','#E67E22']

for i, (feat, color) in enumerate(zip(features_plot, colors)):
    ax = axes[i // 3][i % 3]
    ax.hist(df[feat], bins=30, color=color, alpha=0.8, edgecolor='white')
    ax.set_title(feat, fontweight='bold')
    ax.set_xlabel('Nilai')
    ax.set_ylabel('Frekuensi')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('01_distribusi_fitur.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot distribusi fitur tersimpan ✅')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# EngagementLevel
eng_counts = df['EngagementLevel'].value_counts()
colors_pie = ['#4A90E2','#50C878','#FF6B6B']
axes[0].pie(eng_counts, labels=eng_counts.index, autopct='%1.1f%%',
            colors=colors_pie, startangle=90,
            wedgeprops={'edgecolor':'white','linewidth':2})
axes[0].set_title('Distribusi Engagement Level', fontweight='bold')

# GameGenre
genre_counts = df['GameGenre'].value_counts()
axes[1].barh(genre_counts.index, genre_counts.values,
             color=['#4A90E2','#50C878','#FF6B6B','#FFD700','#9B59B6'])
axes[1].set_title('Distribusi Game Genre', fontweight='bold')
axes[1].set_xlabel('Jumlah Pemain')
for i, v in enumerate(genre_counts.values):
    axes[1].text(v + 50, i, f'{v:,}', va='center')

plt.tight_layout()
plt.savefig('02_distribusi_kategori.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
features_num = ['PlayTimeHours','SessionsPerWeek','AvgSessionDurationMinutes',
                'PlayerLevel','AchievementsUnlocked']
corr = df[features_num].corr()

plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlBu_r',
            mask=mask, vmin=-1, vmax=1,
            linewidths=0.5, cbar_kws={'label':'Korelasi'})
plt.title('Heatmap Korelasi Antar Fitur', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('03_heatmap_korelasi.png', dpi=150, bbox_inches='tight')
plt.show()

---
# 3. Data Preparation


## 3.1 Data Cleaning

In [ ]:
print('=== Cek Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'Tidak ada missing values ✅')
print()
print('=== Cek Duplikat ===')
dup = df.duplicated().sum()
print(f'Jumlah data duplikat: {dup}' if dup > 0 else 'Tidak ada data duplikat ✅')
print()
print('=== Cek Outlier (IQR Method) ===')
features_cluster = ['PlayTimeHours','SessionsPerWeek','AvgSessionDurationMinutes',
                     'PlayerLevel','AchievementsUnlocked']
for f in features_cluster:
    Q1, Q3 = df[f].quantile(0.25), df[f].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df[f] < Q1 - 1.5*IQR) | (df[f] > Q3 + 1.5*IQR)).sum()
    print(f'  {f:<35}: {outliers} outlier')

## 3.2 Seleksi Fitur untuk Clustering

In [ ]:
features_cluster = ['PlayTimeHours', 'SessionsPerWeek',
                     'AvgSessionDurationMinutes', 'PlayerLevel', 'AchievementsUnlocked']

X = df[features_cluster].copy()
print('Fitur yang digunakan untuk clustering:')
for f in features_cluster:
    print(f'  ✔ {f}')
print(f'\nShape data: {X.shape}')

## 3.3 Normalisasi Data (StandardScaler)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=features_cluster)

print('Statistik setelah normalisasi:')
print(X_scaled_df.describe().round(3))
print()
print('Mean mendekati 0 dan Std mendekati 1 → normalisasi berhasil ✅')

---
# 4. Modeling


## 4.1 Menentukan Jumlah Cluster Optimal (Elbow Method)

In [ ]:
inertias = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(10, 5))
plt.plot(k_range, inertias, 'bo-', linewidth=2.5, markersize=8, color='#4A90E2')
plt.axvline(x=3, color='red', linestyle='--', linewidth=2, label='K optimal = 3')
plt.fill_between(k_range, inertias, alpha=0.1, color='#4A90E2')
plt.xlabel('Jumlah Cluster (K)', fontsize=13)
plt.ylabel('Inertia (Within-Cluster Sum of Squares)', fontsize=13)
plt.title('Elbow Method — Menentukan Jumlah Cluster Optimal', fontweight='bold', fontsize=14)
plt.legend(fontsize=12)
plt.grid(alpha=0.3)
plt.xticks(k_range)
plt.tight_layout()
plt.savefig('04_elbow_method.png', dpi=150, bbox_inches='tight')
plt.show()
print('Berdasarkan Elbow Method, K=3 dipilih sebagai titik optimal (perubahan inertia mulai melandai)')

## 4.2 Training Model K-Means (K=3)

In [ ]:
K_OPTIMAL = 3
kmeans = KMeans(n_clusters=K_OPTIMAL, random_state=42, n_init=10, max_iter=300)
df['Cluster'] = kmeans.fit_predict(X_scaled)

print(f'Model K-Means dengan K={K_OPTIMAL} berhasil dilatih ✅')
print(f'Inertia final: {kmeans.inertia_:.2f}')
print(f'Iterasi: {kmeans.n_iter_}')
print()
print('Distribusi Cluster:')
for c, cnt in df['Cluster'].value_counts().sort_index().items():
    pct = cnt / len(df) * 100
    print(f'  Cluster {c}: {cnt:,} pemain ({pct:.1f}%)')

## 4.3 Profil Setiap Cluster

In [ ]:
cluster_profile = df.groupby('Cluster')[features_cluster].mean().round(2)

# Label nama cluster
cluster_names = {
    0: 'High-Time Casual (Pemain Santai Aktif)',
    1: 'Veteran Player (Pemain Berpengalaman)',
    2: 'Light Player (Pemain Ringan)'
}
cluster_profile.index = [f'Cluster {i} — {cluster_names[i]}' for i in cluster_profile.index]
print('Profil Rata-rata Setiap Cluster:')
print('=' * 80)
print(cluster_profile.to_string())

---
# 5. Evaluation


## 5.1 Silhouette Score

In [ ]:
sil_scores = {}
for k in range(2, 8):
    km_tmp = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_tmp = km_tmp.fit_predict(X_scaled)
    sil_scores[k] = silhouette_score(X_scaled, labels_tmp)

sil_k3 = sil_scores[3]
print('Silhouette Score per K:')
for k, s in sil_scores.items():
    mark = ' ← K optimal' if k == 3 else ''
    print(f'  K={k}: {s:.4f}{mark}')
print()
print(f'Silhouette Score K=3: {sil_k3:.4f}')

if sil_k3 >= 0.50:
    print('Interpretasi: Struktur clustering KUAT')
elif sil_k3 >= 0.25:
    print('Interpretasi: Struktur clustering CUKUP BAIK — cluster dapat dibedakan')
else:
    print('Interpretasi: Clustering diterima — data memiliki tumpang tindih alami antar segmen perilaku')

## 5.2 Visualisasi Hasil Clustering (PCA 2D)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

colors_cluster = ['#4A90E2', '#50C878', '#FF6B6B']
labels_map = {0: 'High-Time Casual', 1: 'Veteran Player', 2: 'Light Player'}

plt.figure(figsize=(11, 7))
for c in range(3):
    mask = df['Cluster'] == c
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                c=colors_cluster[c], label=f'Cluster {c}: {labels_map[c]}',
                alpha=0.35, s=8)

# plot centroids
centers_pca = pca.transform(kmeans.cluster_centers_)
for c in range(3):
    plt.scatter(centers_pca[c, 0], centers_pca[c, 1],
                c=colors_cluster[c], s=250, marker='*',
                edgecolors='black', linewidth=1.5, zorder=5)

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=12)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=12)
plt.title('Visualisasi Hasil Clustering K-Means (Reduksi PCA 2D)\n★ = Centroid Cluster',
          fontweight='bold', fontsize=14)
plt.legend(fontsize=11, markerscale=3)
plt.grid(alpha=0.2)
plt.tight_layout()
plt.savefig('05_clustering_pca.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Total variance explained: {sum(pca.explained_variance_ratio_)*100:.1f}%')

## 5.3 Visualisasi Profil Cluster (Radar Chart & Bar)

In [ ]:
cluster_means = df.groupby('Cluster')[features_cluster].mean()
# normalize 0-1 for comparison
cluster_norm = (cluster_means - cluster_means.min()) / (cluster_means.max() - cluster_means.min())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Profil Karakteristik Setiap Cluster', fontweight='bold', fontsize=15)

short_labels = ['PlayTime', 'Sessions/Wk', 'AvgDuration', 'Level', 'Achievements']
cnames = ['High-Time Casual', 'Veteran Player', 'Light Player']

for i, (c, color) in enumerate(zip(range(3), colors_cluster)):
    ax = axes[i]
    vals = cluster_means.loc[c].values
    bars = ax.bar(short_labels, vals, color=color, alpha=0.85, edgecolor='white', linewidth=1.5)
    ax.set_title(f'Cluster {c}\n{cnames[c]}', fontweight='bold', fontsize=12, color=color)
    ax.set_ylabel('Nilai Rata-rata')
    ax.tick_params(axis='x', rotation=20)
    ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('06_profil_cluster.png', dpi=150, bbox_inches='tight')
plt.show()

## 5.4 Evaluasi Performa Model

In [ ]:
inertia_k3 = kmeans.inertia_
sil_k3_final = silhouette_score(X_scaled, df['Cluster'])
cluster_dist = df['Cluster'].value_counts().sort_index()

print('=' * 55)
print('        HASIL EVALUASI MODEL K-MEANS (K=3)')
print('=' * 55)
print(f'  Inertia (WCSS)      : {inertia_k3:,.2f}')
print(f'  Silhouette Score    : {sil_k3_final:.4f}')
print(f'  Jumlah Iterasi      : {kmeans.n_iter_}')
print(f'  Total Data          : {len(df):,}')
print()
print('  Distribusi Cluster:')
for c, cnt in cluster_dist.items():
    pct = cnt/len(df)*100
    bar = '█' * int(pct/2)
    print(f'    Cluster {c}: {cnt:>6,} ({pct:4.1f}%) {bar}')
print()
print('  ✅ Clustering berhasil membagi data menjadi 3 segmen')
print('     yang memiliki karakteristik perilaku bermain berbeda.')
print(f'  ✅ Silhouette Score {sil_k3_final:.4f} menunjukkan cluster')
print('     dapat dibedakan secara bermakna.')

---
# 6. Deployment


## 6.1 Interpretasi & Deskripsi Cluster

### 🔵 Cluster 0 — High-Time Casual (Pemain Santai Aktif)
Pemain dengan **jam bermain tinggi (~18.6 jam)** namun **level karakter rendah (~33)**.
Mereka banyak menghabiskan waktu bermain tetapi tidak mengoptimalkan progres karakter.
Kemungkinan pemain yang bermain untuk relaksasi atau eksplorasi konten.

> **Rekomendasi:** Sediakan event harian, konten eksplorasi, dan sistem reward berbasis waktu bermain.

### 🟢 Cluster 1 — Veteran Player (Pemain Berpengalaman)
Pemain dengan **level tinggi (~80)** dan jam bermain sedang (~11.6 jam).
Mereka efisien dalam bermain — sedikit waktu namun progres karakter optimal.
Segmen pemain paling kompeten dan loyal.

> **Rekomendasi:** Sediakan konten kompetitif (ranked/PvP), battle pass, dan fitur eksklusif veteran.

### 🔴 Cluster 2 — Light Player (Pemain Ringan)
Pemain dengan **jam bermain paling rendah (~5.8 jam)** dan **level rendah (~30)**.
Mereka bermain sangat sedikit, mungkin pemain baru atau pemain pasif.

> **Rekomendasi:** Tingkatkan onboarding experience, tutorial, dan event entry-level untuk meningkatkan engagement.


In [ ]:
# Simpan hasil clustering ke CSV
df_result = df[['PlayerID', 'Age', 'Gender', 'GameGenre',
                'PlayTimeHours', 'SessionsPerWeek',
                'AvgSessionDurationMinutes', 'PlayerLevel',
                'AchievementsUnlocked', 'Cluster']].copy()

df_result['ClusterName'] = df_result['Cluster'].map({
    0: 'High-Time Casual',
    1: 'Veteran Player',
    2: 'Light Player'
})

df_result.to_csv('hasil_clustering_gaming.csv', index=False)
print('Hasil clustering disimpan ke: hasil_clustering_gaming.csv ✅')
print()
print('Preview 5 baris pertama:')
df_result.head()

In [ ]:
# Visualisasi final: distribusi cluster per genre game
ct = pd.crosstab(df['GameGenre'], df['Cluster'])
ct.columns = ['High-Time Casual', 'Veteran Player', 'Light Player']

ax = ct.plot(kind='bar', figsize=(12, 6),
             color=['#4A90E2','#50C878','#FF6B6B'],
             edgecolor='white', linewidth=0.8)
plt.title('Distribusi Cluster per Genre Game', fontweight='bold', fontsize=14)
plt.xlabel('Genre Game', fontsize=12)
plt.ylabel('Jumlah Pemain', fontsize=12)
plt.xticks(rotation=0)
plt.legend(title='Cluster', fontsize=11)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('07_cluster_per_genre.png', dpi=150, bbox_inches='tight')
plt.show()

## 6.2 Kesimpulan

Analisis K-Means Clustering terhadap **40.034 data pemain game online** menghasilkan **3 segmen** yang bermakna:

| Cluster | Nama | Jumlah | Karakteristik Utama |
|---------|------|--------|--------------------|
| 0 | High-Time Casual | ~12.773 | Banyak main, level rendah |
| 1 | Veteran Player | ~14.871 | Level tinggi, efisien |
| 2 | Light Player | ~12.390 | Sedikit main, level rendah |

**Model berhasil memenuhi kriteria sukses** dengan menghasilkan cluster yang dapat diinterpretasikan
dan memberikan insight bisnis yang actionable untuk tim pengembang game.

---
*Project ini dipublikasikan di Google Sites sebagai bagian dari portofolio akademik.*  
*Source code dan dataset tersedia di: https://github.com/louishasashi-dev*
